# 第 1 週 實作｜極限與連續（Colab notebook）

**目標**：把理論課的六個概念**親手跑出來、畫出來**——
1. 數值探極限（`sin x / x`）
2. 夾擠定理視覺化
3. 單邊極限與不連續
4. 二分逼近 × 中間值定理（IVT）
5. 漸近線視覺化（水平／鉛直）
6. 分段函數連不連續

**用法**：上傳到 [Google Colab](https://colab.research.google.com/) 或本機 Jupyter，由上往下逐格執行。
標「`# TODO 學生練習`」的格子留給學生填。

> 圖表標籤用英文/數學符號以避免中文變豆腐字；中文都放在說明格。

In [ ]:
# === 環境設定(先跑這格)===
import math
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True
print("環境就緒, numpy", np.__version__)

### (選用)讓圖表顯示中文

預設圖表用英文標籤,避免中文變「豆腐字」。若你在 Colab 想要中文座標/標題,
把下一格的註解取消再執行(只需一次),之後的圖就能顯示中文。

In [ ]:
# 想要中文圖標時,取消以下註解執行(Colab 適用;本機 Jupyter 需自備 CJK 字型)
# !apt-get -qq install fonts-noto-cjk > /dev/null
# import matplotlib
# matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# matplotlib.rcParams['axes.unicode_minus'] = False
print("預設英文標籤;要中文請見上一格說明")

## Lab 1｜數值探極限

`f(x) = sin(x)/x` 在 `x = 0` 是 `0/0`(沒定義),但把 x 從兩邊逼近 0,值會朝某個數收斂。
先看招牌例子,再用通用工具 `numeric_limit` 探索別的極限。

In [ ]:
# 招牌例子:lim(x->0) sin(x)/x = ?
print("  x         sin(x)/x")
for x in [1, 0.5, 0.1, 0.01, 0.001, 0.0001]:
    print(f"{x:<9} {math.sin(x)/x:.8f}")
# 觀察:x 越靠近 0,值越靠近 1 -> 極限 = 1

In [ ]:
def numeric_limit(f, a, side="both", n=6):
    """從 a 的左/右用越來越近的點觀察 f 的走向。side: 'left' / 'right' / 'both'"""
    hs = [10**-k for k in range(1, n + 1)]
    if side in ("right", "both"):
        print(f"從右邊逼近 x={a}:")
        for h in hs:
            print(f"  x={a+h:<12.6g}  f(x)={f(a+h):.8f}")
    if side in ("left", "both"):
        print(f"從左邊逼近 x={a}:")
        for h in hs:
            print(f"  x={a-h:<12.6g}  f(x)={f(a-h):.8f}")

# 示範:lim(x->3) (x^2-9)/(x-3),理論值 6
numeric_limit(lambda x: (x**2 - 9) / (x - 3), 3)

In [ ]:
# TODO 學生練習:用 numeric_limit 觀察下面兩個極限,先猜答案,再回去用手算(有理化/共軛)驗證
# (1) lim(x->0) (1 - cos x) / x^2
# (2) lim(x->0) (sqrt(1+x) - 1) / x

# numeric_limit(lambda x: (1 - math.cos(x)) / x**2, 0)
# numeric_limit(lambda x: (math.sqrt(1 + x) - 1) / x, 0)

# 參考答案(先猜再看):(1) 0.5   (2) 0.5

## Lab 2｜夾擠定理視覺化

`-x^2 <= x^2 sin(1/x) <= x^2`。兩側都 -> 0,中間被「夾」著,只能一起 -> 0。
`sin(1/x)` 自己在 0 附近劇烈震盪、沒有極限,但乘上 `x^2` 把振幅壓成 0,極限就出現了。

In [ ]:
x = np.linspace(-0.5, 0.5, 3000)
x = x[x != 0]
plt.plot(x,  x**2, '--', label='y = x^2  (upper bound)')
plt.plot(x, -x**2, '--', label='y = -x^2 (lower bound)')
plt.plot(x, x**2 * np.sin(1 / x), lw=1, label='y = x^2 sin(1/x)')
plt.axhline(0, color='gray', lw=0.8); plt.axvline(0, color='gray', lw=0.8)
plt.title('Squeeze Theorem:  x^2 sin(1/x) -> 0  as x -> 0')
plt.legend(); plt.show()

## Lab 3｜單邊極限與不連續

`f(x) = |x| / x`:x>0 時是 +1,x<0 時是 -1。左右極限不相等 -> 雙邊極限**不存在**(跳躍不連續)。

In [ ]:
f = lambda x: np.abs(x) / x
xl = np.linspace(-2, -1e-3, 500)
xr = np.linspace(1e-3, 2, 500)
plt.plot(xl, f(xl), label='x < 0  ->  -1')
plt.plot(xr, f(xr), label='x > 0  ->  +1')
plt.plot(0, -1, 'o', mfc='white', mec='C0')   # 空心點:此處不連續
plt.plot(0,  1, 'o', mfc='white', mec='C1')
plt.axhline(0, color='gray', lw=0.8); plt.axvline(0, color='gray', lw=0.8)
plt.title('Jump discontinuity: left limit=-1, right limit=+1  ->  limit DNE')
plt.legend(); plt.ylim(-2, 2); plt.show()

print("左極限 ~", f(-1e-9), "   右極限 ~", f(1e-9))

## Lab 4｜二分逼近 × 中間值定理(IVT)

**IVT**:`f` 在 `[a,b]` 連續且 `f(a)`、`f(b)` **異號** => `(a,b)` 內至少有一根(只保證存在)。
二分法就把這個「存在」一步步夾成「近似值」——每次取中點、留下仍然異號的半邊。

In [ ]:
def bisect(f, a, b, tol=1e-10, verbose=True):
    assert f(a) * f(b) < 0, "f(a) 與 f(b) 必須異號(這正是 IVT 的前提)"
    step = 0
    while b - a > tol:
        m = (a + b) / 2
        if f(a) * f(m) <= 0:
            b = m
        else:
            a = m
        step += 1
        if verbose and step <= 8:
            print(f"step {step:2d}:  區間=[{a:.8f}, {b:.8f}]  中點 f(m)={f((a+b)/2):+.2e}")
    return (a + b) / 2

f = lambda x: x**3 - x - 1
print("f(1) =", f(1), "  f(2) =", f(2), " -> 異號,IVT 保證 (1,2) 內有根\n")
root = bisect(f, 1, 2)
print("\n近似根 x ~", root, "   驗證 f(root) =", f(root))

In [ ]:
# TODO 學生練習:證明 cos(x) = x 在 (0,1) 有解,並用 bisect 找出來
# 提示:令 g(x) = cos(x) - x,先檢查 g(0)、g(1) 是否異號(對應例題 13)

# g = lambda x: math.cos(x) - x
# print("g(0) =", g(0), "  g(1) =", g(1))
# print("root ~", bisect(g, 0, 1))
# 參考答案:root ~ 0.7390851

## Lab 5｜漸近線視覺化

把「趨向無窮」畫出來:有理函數會貼近**水平漸近線**(x→±∞)與**鉛直漸近線**(分母=0)。

In [ ]:
# 水平漸近線:f(x)=(3x^2+2x-1)/(2x^2-x+5) -> y = 1.5
f = lambda x: (3*x**2 + 2*x - 1) / (2*x**2 - x + 5)
x = np.linspace(-20, 20, 2000)
plt.plot(x, f(x), label='f(x)')
plt.axhline(1.5, color='C1', ls='--', label='horizontal asymptote y=1.5')
plt.title('Horizontal asymptote: f -> 3/2'); plt.legend(); plt.ylim(-1, 4); plt.show()

# 鉛直漸近線:g(x)=x/(x-2) -> x=2 兩側衝 +-inf
g = lambda x: x/(x - 2)
xl = np.linspace(-3, 1.98, 800); xr = np.linspace(2.02, 7, 800)
plt.plot(xl, g(xl), 'C0'); plt.plot(xr, g(xr), 'C0', label='g(x)')
plt.axvline(2, color='C3', ls='--', label='vertical asymptote x=2')
plt.axhline(1, color='C2', ls=':', label='horizontal asymptote y=1')
plt.title('Vertical asymptote x=2'); plt.legend(); plt.ylim(-15, 15); plt.show()

## Lab 6｜分段函數連不連續

待定係數在做的事,畫出來就懂:選對參數兩段**接得起來**(連續),選錯就**斷一個縫**。

In [ ]:
# f(x)= a*x+3 (x<1) 接 x^2 (x>=1);連續需要 a=-2(接縫 x=1 兩邊相等)
def piece(a, title):
    xl = np.linspace(-1, 1, 400); xr = np.linspace(1, 3, 400)
    plt.plot(xl, a*xl + 3, label=f'a*x+3 (a={a})')
    plt.plot(xr, xr**2, label='x^2')
    plt.axvline(1, color='gray', ls=':'); plt.title(title); plt.legend(fontsize=8)

plt.subplot(1, 2, 1); piece(-2, 'a=-2: continuous')
plt.subplot(1, 2, 2); piece(0,  'a=0: jump')
plt.tight_layout(); plt.show()

## 收尾 · 與筆試的連結

| 這格實作 | 對應觀念 |
|---|---|
| Lab 1 數值表 | 極限直覺、`sin x / x = 1`(觀念 11) |
| Lab 2 夾擠圖 | 夾擠定理(觀念 10) |
| Lab 3 `|x|/x` | 單邊極限、雙邊不存在(觀念 12) |
| Lab 4 二分 × IVT | 中間值定理(觀念 17) |
| Lab 5 漸近線 | 趨向無窮、水平/鉛直漸近線(觀念 5、8) |
| Lab 6 分段連續 | 連續與待定係數(觀念 14、16) |

### 進階徽章(選做)
1. 用 `numeric_limit` 找一個「左右極限存在但不相等」的例子,並畫出來。
2. 把 Lab 4 的二分法改成**回傳迭代次數**,觀察誤差如何每步大約減半(預告第 3 週牛頓法)。
3. 用 `sympy`:`sp.limit(sp.sin(x)/x, x, 0)` 驗證你手算的極限。